##### Step1 Filter the interaction data from NPInter and separately obtain the LPI for human and mouse.

In [1]:
import pandas as pd

input_file = "../../data/raw/lncRNA_interaction.txt"
output_human = "../../data/LPI/human/npinter_lpi.csv"
output_mouse = "../../data/LPI/mouse/npinter_lpi.csv"

# indices to keep: (gene_name, gene_id, protein_name, uniprot_id, tissue_or_cellline)
keep_indices = [1, 2, 4, 5, 11]

# final output header
header = ['gene_name', 'gene_id', 'identifier', 'protein', 'uniprot_id', 'tissue_or_cellline']

# read input file
df = pd.read_csv(input_file, sep="\t", header=None, dtype=str)

# filter rows: lncRNA - protein - binding
df = df[(df[3] == "lncRNA") & (df[6] == "protein") & (df[13] == "binding")]

# fix gene_id
df.iloc[:, 2] = df.iloc[:, 2].replace("NOCODE", "-")

# select needed columns
filtered = df.iloc[:, keep_indices].copy()
filtered.columns = ['gene_name', 'gene_id', 'protein', 'uniprot_id', 'tissue_or_cellline']

# construct identifier column
filtered["identifier"] = filtered.apply(
    lambda x: x["gene_id"] if x["gene_id"] != "-" else x["gene_name"], axis=1
)

# reorder columns
filtered = filtered[['gene_name', 'gene_id', 'identifier', 'protein', 'uniprot_id']]

# remove self-loops (gene_name == protein)
filtered = filtered[~((filtered['gene_name'] != "") & (filtered['protein'] != "") & 
                     (filtered['gene_name'] == filtered['protein']))]

# split by species
human_lpi = filtered[df[10] == "Homo sapiens"].drop_duplicates()
mouse_lpi = filtered[df[10] == "Mus musculus"].drop_duplicates()

# save to CSV
human_lpi.to_csv(output_human, index=False, encoding='utf-8')
mouse_lpi.to_csv(output_mouse, index=False, encoding='utf-8')

print("Done. Generated npinter_lpi.csv for human and mouse with columns: "
      "gene_name, gene_id, identifier, protein_name, uniprot_id, tissue_or_cellline.")


/tmp/ipykernel_8161/3455257612.py:39: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  human_lpi = filtered[df[10] == "Homo sapiens"].drop_duplicates()
/tmp/ipykernel_8161/3455257612.py:40: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  mouse_lpi = filtered[df[10] == "Mus musculus"].drop_duplicates()


Done. Generated npinter_lpi.csv for human and mouse with columns: gene_name, gene_id, identifier, protein_name, uniprot_id, tissue_or_cellline.


##### Step2：Fix the LPI data.

Step 2.1 Replace the transcript IDs with their corresponding gene IDs.

In [2]:
# Human
# -----------------------------
# Paths
# -----------------------------
mapping_file_noncode6 = "../../reference_lncRNA/human/transcript/NONCODEv6_human_hg38_lncRNA_trans.txt"
mapping_file_noncode5 = "../../reference_lncRNA/human/transcript/NONCODEv5_human_hg38_lncRNA_trans.txt"
out_file = "human_lpi_id_fixed.csv"

# Transcript-like ID prefixes to repair
transcript_prefixes = ("NONHSAT",)

# -----------------------------
# Helper: load transcript->gene mapping
# Assumes each line has at least two columns:
#   col0 = gene_id, col1 = transcript_id
# Auto-detects delimiter (comma/tab/whitespace) via engine='python'.
# -----------------------------
def load_mapping(path: str) -> dict:
    df = pd.read_csv(
        path,
        sep=None,                # auto-detect delimiter
        engine="python",
        header=None,
        usecols=[0, 1],          # [gene_id, transcript_id]
        names=["gene_id", "transcript_id"],
        dtype=str
    )
    # normalize strings
    df = df.dropna(subset=["gene_id", "transcript_id"])
    df["gene_id"] = df["gene_id"].str.strip()
    df["transcript_id"] = df["transcript_id"].str.strip()
    # build transcript -> gene mapping (v6/v5 priority handled outside)
    return dict(zip(df["transcript_id"], df["gene_id"]))

# Load mappings: v6 has higher priority than v5
map_v6 = load_mapping(mapping_file_noncode6)
map_v5 = load_mapping(mapping_file_noncode5)

# -----------------------------
# Load LPI table
# Keep strings to avoid unintended type casting
# -----------------------------

# Normalize gene_id string for testing and lookups
gid_norm = human_lpi["gene_id"].fillna("").astype(str).str.strip()

# Identify rows that look like transcript IDs (to be repaired)
mask_tx_like = gid_norm.str.startswith(transcript_prefixes)

# Map transcript -> gene via v6 (priority) and v5 (fallback) only on those rows
mapped_v6 = gid_norm.where(mask_tx_like).map(map_v6)
mapped_v5 = gid_norm.where(mask_tx_like).map(map_v5)

# Start with original gene_id and apply repairs
new_gene_id = human_lpi["gene_id"].copy()

# Apply v6 where available
mask_v6_hit = mapped_v6.notna()
new_gene_id.loc[mask_v6_hit] = mapped_v6.loc[mask_v6_hit].values

# Apply v5 where v6 missed but v5 hit
mask_v5_hit = (~mask_v6_hit) & mapped_v5.notna()
new_gene_id.loc[mask_v5_hit] = mapped_v5.loc[mask_v5_hit].values

# Update gene_id in the dataframe
human_lpi["gene_id"] = new_gene_id

# Rows actually repaired (either v6 or v5 hit)
mask_repaired = mask_v6_hit | mask_v5_hit

# Keep identifier in sync with repaired gene_id (same behavior as original script)
human_lpi.loc[mask_repaired, "identifier"] = human_lpi.loc[mask_repaired, "gene_id"]

# Save
human_lpi.to_csv(out_file, index=False)


In [3]:
# Mouse
# -----------------------------
# Paths
# -----------------------------
mapping_file_noncode5 = "../../reference_lncRNA/mouse/transcript/NONCODEv5_mouse_mm10_lncRNA_trans.txt"
out_file = "mouse_lpi_id_fixed.csv"

transcript_prefixes = ("NONMMUT",)

# -----------------------------
# Helper: load transcript->gene mapping
# Assumes each line has at least two columns:
#   col0 = gene_id, col1 = transcript_id
# Auto-detects delimiter (comma/tab/whitespace) via engine='python'.
# -----------------------------
def load_mapping(path: str) -> dict:
    df = pd.read_csv(
        path,
        sep=None,                # auto-detect delimiter
        engine="python",
        header=None,
        usecols=[0, 1],          # [gene_id, transcript_id]
        names=["gene_id", "transcript_id"],
        dtype=str
    )
    # normalize strings
    df = df.dropna(subset=["gene_id", "transcript_id"])
    df["gene_id"] = df["gene_id"].str.strip()
    df["transcript_id"] = df["transcript_id"].str.strip()
    # build transcript -> gene mapping (v6/v5 priority handled outside)
    return dict(zip(df["transcript_id"], df["gene_id"]))

# Load mappings
map_v5 = load_mapping(mapping_file_noncode5)

# -----------------------------
# Load LPI table
# Keep strings to avoid unintended type casting
# -----------------------------

# Normalize gene_id string for testing and lookups
gid_norm = mouse_lpi["gene_id"].fillna("").astype(str).str.strip()

# Identify rows that look like transcript IDs (to be repaired)
mask_tx_like = gid_norm.str.startswith(transcript_prefixes)

# Map transcript -> gene via v5
mapped_v5 = gid_norm.where(mask_tx_like).map(map_v5)

# Start with original gene_id and apply repairs
new_gene_id = mouse_lpi["gene_id"].copy()

# Apply v5
mask_v5_hit = mapped_v5.notna()
new_gene_id.loc[mask_v5_hit] = mapped_v5.loc[mask_v5_hit].values

# Update gene_id in the dataframe
mouse_lpi["gene_id"] = new_gene_id

# Rows actually repaired
mask_repaired = mask_v5_hit

# Keep identifier in sync with repaired gene_id (same behavior as original script)
mouse_lpi.loc[mask_repaired, "identifier"] = mouse_lpi.loc[mask_repaired, "gene_id"]

# Save
mouse_lpi.to_csv(out_file, index=False)


Step2.2 Replace invalid identifier with gene_name

In [4]:
# Human
import pandas as pd
import os
import re

# ----------------------------
# Paths (adjust as needed)
# ----------------------------
ensembl_dir = "../../reference_lncRNA/human/bed/ensembl/"
# human_lpi = pd.read_csv('human_lpi_id_fixed.csv')

# ----------------------------
# Load NONCODE gene_id lists (only gene_id column is needed)
# NONCODE BED columns: chr, start, end, gene_id, score, strand
# ----------------------------
noncodev5_ids = pd.read_csv(
    '../../reference_lncRNA/human/bed/NONCODEv5_hg38.lncRNAGene.bed',
    sep='\t', header=None, usecols=[3], names=['gene_id']
)['gene_id']

noncodev6_ids = pd.read_csv(
    '../../reference_lncRNA/human/bed/NONCODEv6_hg38.lncRNAGene.bed',
    sep='\t', header=None, usecols=[3], names=['gene_id']
)['gene_id']

# ----------------------------
# Collect Ensembl gene_id from all BED files in the directory
# Ensembl BED columns: chr, start, end, gene_name, gene_id, strand
# ----------------------------
def extract_version(filename: str) -> int:
    """
    Extract Ensembl GRCh38 version number from filename (e.g., '...GRCh38.<n>.bed').
    Returns -1 if not matched.
    """
    m = re.search(r'GRCh38\.(\d+)\.bed', filename)
    return int(m.group(1)) if m else -1

bed_files = [f for f in os.listdir(ensembl_dir) if f.endswith(".bed")]
# Sorting not strictly required for validity checking, but kept for consistency
bed_files_sorted = sorted(bed_files, key=extract_version, reverse=True)

ensembl_ids_list = []
for bed_file in bed_files_sorted:
    bed_path = os.path.join(ensembl_dir, bed_file)
    # Read only the gene_id column (index 4)
    gid = pd.read_csv(bed_path, sep='\t', header=None, usecols=[4], names=['gene_id'])['gene_id']
    ensembl_ids_list.append(gid)

ensembl_ids = pd.concat(ensembl_ids_list, ignore_index=True) if ensembl_ids_list else pd.Series([], dtype=object)

# ----------------------------
# Build a set of valid gene IDs across all sources
# ----------------------------
def to_id_set(series: pd.Series) -> set:
    """
    Normalize a Series to a set of non-empty string IDs:
    - drop NaN
    - strip spaces
    - drop empty strings
    """
    s = series.dropna().astype(str).str.strip()
    s = s[s != ""]
    return set(s)

valid_ids = to_id_set(noncodev6_ids) | to_id_set(noncodev5_ids) | to_id_set(ensembl_ids)

# ----------------------------
# Determine invalid IDs and update 'identifier' accordingly
# Rules:
# - If gene_id is NOT in valid_ids (or is null/empty) -> treat as invalid
# - For invalid rows: identifier := gene_name (gene_name!=-,gene_name is valid)
# - For valid rows: identifier remains unchanged
# - Filter out rows with invalid gene_id and invalid gene_name
# ----------------------------
# ----------------------------
# Determine valid and invalid gene_id
# ----------------------------
gene_id_raw = human_lpi['gene_id']
gene_id_norm = gene_id_raw.astype(str).str.strip()

mask_valid_id = gene_id_norm.isin(valid_ids)
mask_invalid_id = gene_id_raw.isna() | (gene_id_norm == "") | (~gene_id_norm.isin(valid_ids))

# ----------------------------
# For invalid gene_id, check gene_name validity
# ----------------------------
gene_name_norm = human_lpi['gene_name'].astype(str).str.strip()
mask_valid_name = (gene_name_norm != "") & (gene_name_norm != "-")

# Rows to replace identifier with gene_name
mask_replace = mask_invalid_id & mask_valid_name

# Rows to drop: both ID invalid and name invalid
mask_drop = mask_invalid_id & (~mask_valid_name)

# Apply replacement
human_lpi.loc[mask_replace, 'identifier'] = human_lpi.loc[mask_replace, 'gene_name']

# Drop completely invalid rows
human_lpi = human_lpi.loc[~mask_drop].copy()

# save the updated table
human_lpi.to_csv('./human_lpi_fixed.csv', index=False)


In [5]:
# Mouse
import pandas as pd
import os
import re

# ----------------------------
# Paths (adjust as needed)
# ----------------------------
ensembl_dir = "../../reference_lncRNA/mouse/bed/ensembl/"
#mouse_lpi = pd.read_csv('mouse_lpi_id_fixed.csv')

# ----------------------------
# Load NONCODE gene_id lists (only gene_id column is needed)
# NONCODE BED columns: chr, start, end, gene_id, score, strand
# ----------------------------
noncodev5_ids = pd.read_csv(
    '../../reference_lncRNA/mouse/bed/NONCODEv5_mm10.lncRNAGene.bed',
    sep='\t', header=None, usecols=[3], names=['gene_id']
)['gene_id']

noncodev6_ids = pd.read_csv(
    '../../reference_lncRNA/mouse/bed/NONCODEv6_mm10.lncRNAGene.bed',
    sep='\t', header=None, usecols=[3], names=['gene_id']
)['gene_id']

# ----------------------------
# Collect Ensembl gene_id from all BED files in the directory
# Ensembl BED columns: chr, start, end, gene_name, gene_id, strand
# ----------------------------
def extract_version(filename: str) -> int:
    """
    Extract Ensembl GRCm38 version number from filename (e.g., '...GRCm38.<n>.bed').
    Returns -1 if not matched.
    """
    m = re.search(r'GRCm38\.(\d+)\.bed', filename)
    return int(m.group(1)) if m else -1

bed_files = [f for f in os.listdir(ensembl_dir) if re.match(r"Mus_musculus\.GRCm38\.\d+\.bed$", f)]
# Sorting not strictly required for validity checking, but kept for consistency
bed_files_sorted = sorted(bed_files, key=extract_version, reverse=True)

ensembl_ids_list = []
for bed_file in bed_files_sorted:
    bed_path = os.path.join(ensembl_dir, bed_file)
    # Read only the gene_id column (index 4)
    gid = pd.read_csv(bed_path, sep='\t', header=None, usecols=[4], names=['gene_id'])['gene_id']
    ensembl_ids_list.append(gid)

ensembl_ids = pd.concat(ensembl_ids_list, ignore_index=True) if ensembl_ids_list else pd.Series([], dtype=object)

# ----------------------------
# Build a set of valid gene IDs across all sources
# ----------------------------
def to_id_set(series: pd.Series) -> set:
    """
    Normalize a Series to a set of non-empty string IDs:
    - drop NaN
    - strip spaces
    - drop empty strings
    """
    s = series.dropna().astype(str).str.strip()
    s = s[s != ""]
    return set(s)

valid_ids = to_id_set(noncodev6_ids) | to_id_set(noncodev5_ids) | to_id_set(ensembl_ids)

# ----------------------------
# Determine invalid IDs and update 'identifier' accordingly
# Rules:
# - If gene_id is NOT in valid_ids (or is null/empty) -> treat as invalid
# - For invalid rows: identifier := gene_name (gene_name!=-,gene_name is valid)
# - For valid rows: identifier remains unchanged
# - Filter out rows with invalid gene_id and invalid gene_name
# ----------------------------
# ----------------------------
# Determine valid and invalid gene_id
# ----------------------------
gene_id_raw = mouse_lpi['gene_id']
gene_id_norm = gene_id_raw.astype(str).str.strip()

mask_valid_id = gene_id_norm.isin(valid_ids)
mask_invalid_id = gene_id_raw.isna() | (gene_id_norm == "") | (~gene_id_norm.isin(valid_ids))

# ----------------------------
# For invalid gene_id, check gene_name validity
# ----------------------------
gene_name_norm = mouse_lpi['gene_name'].astype(str).str.strip()
mask_valid_name = (gene_name_norm != "") & (gene_name_norm != "-")

# Rows to replace identifier with gene_name
mask_replace = mask_invalid_id & mask_valid_name

# Rows to drop: both ID invalid and name invalid
mask_drop = mask_invalid_id & (~mask_valid_name)

# Apply replacement
mouse_lpi.loc[mask_replace, 'identifier'] = mouse_lpi.loc[mask_replace, 'gene_name']

# save the updated table
mouse_lpi.to_csv('./mouse_lpi_fixed.csv', index=False)

#### Step 3 Construct LPI networks.

In [6]:
# Human

human_lpi = pd.read_csv("human_lpi_fixed.csv")

human_lpi["gene_name"] = (
    human_lpi["gene_name"]
    .astype(str)
    .str.replace("‐", "-", regex=False)
)

def concat_ignore_dash(series):
    vals = [str(v).strip() for v in series if pd.notna(v) and str(v).strip() != "-"]
    seen = set()
    out = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return ";".join(out)

lncRNA = human_lpi[['identifier','gene_name','gene_id']].drop_duplicates()
lncRNA = (
    lncRNA.groupby(["identifier"], as_index=False)
      .agg({
          "gene_name": concat_ignore_dash,
          "gene_id": concat_ignore_dash,
      })
)
lncRNA.to_csv("../../data/LPI/human/lncRNA.csv", index=False)

protein = human_lpi[['protein', 'uniprot_id']].drop_duplicates()
protein.to_csv("../../data/LPI/human/protein.csv", index=False)

human_lpi = human_lpi.drop(['gene_id', 'gene_name', 'uniprot_id'], axis=1)

human_lpi = human_lpi.drop_duplicates();
human_lpi.to_csv("../../data/LPI/human/lpi.csv",index=False)


In [7]:
# Mouse

mouse_lpi = pd.read_csv("mouse_lpi_fixed.csv")

mouse_lpi["gene_name"] = (
    mouse_lpi["gene_name"]
    .astype(str)
    .str.replace("‐", "-", regex=False)
)

def concat_ignore_dash(series):
    vals = [str(v).strip() for v in series if pd.notna(v) and str(v).strip() != "-"]
    seen = set()
    out = []
    for v in vals:
        if v not in seen:
            seen.add(v)
            out.append(v)
    return ";".join(out)

lncRNA = mouse_lpi[['identifier', 'gene_name','gene_id']].drop_duplicates()
lncRNA = (
    lncRNA.groupby(["identifier"], as_index=False)
      .agg({
          "gene_name": concat_ignore_dash,
          "gene_id": concat_ignore_dash
      })
)
lncRNA.to_csv("../../data/LPI/mouse/lncRNA.csv", index=False)

protein = mouse_lpi[['protein', 'uniprot_id']].drop_duplicates()
protein.to_csv("../../data/LPI/mouse/protein.csv", index=False)

mouse_lpi = mouse_lpi.drop(['gene_id', 'gene_name', 'uniprot_id'], axis=1)

mouse_lpi = mouse_lpi.drop_duplicates();
mouse_lpi.to_csv("../../data/LPI/mouse/lpi.csv",index=False)
